# Loss Functions


Most figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL), Chapter 5, unless another source is cited. Code cells adapted from the
UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Outline

So far we have *linear regression*, *shallow* and *deep neural networks*, each
with parameters $\boldsymbol{\phi}$ chosen to minimize a *loss function*
$L[\boldsymbol{\phi}]$: a single number measuring the mismatch between
$\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]$ and the targets $y_i$. Where do
loss functions come from?

- Probability models of the output
- Maximum likelihood, and the recipe for loss functions
- Example 1: univariate regression → least squares
- Example 2: binary classification → binary cross-entropy
- Example 3: multiclass classification → cross-entropy
- Other types of data; multiple outputs
- Cross entropy and the KL divergence

# Probability models

We need a loss function that works for each kind of prediction we want to
make: real-valued outputs, binary labels and multiple classes. The unifying
idea is to make the model predict a *probability distribution* over the
output rather than a single value. DS701 covered probability and maximum
likelihood, so this section moves quickly; UDL Appendix C has the review.


## Three kinds of prediction

**Regression**

![](figs/03-Loss-Functions/height-age-data.png)


**Binary classification**

![](figs/03-Loss-Functions/binary-classification-spam-data.png)


**Multiclass classification**

![](figs/03-Loss-Functions/multiclass-classification-tail-data.png)


We need a loss function that works for each of these output types.

## Fitting a model vs. fitting a probability model


![A fitted model (top) and a conditional probability distribution over $y$ at one value of $x$ (bottom).](figs/03-Loss-Functions/height-age-model-fit.png)

![](figs/03-Loss-Functions/conditional-continuous-1.png)


So far, we thought about fitting a model to the data…

Alternatively, we can think about fitting a *probability model* to the data.

$$
\mathrm{Pr}(y \mid x) \text{ at one value of } x
$$

Why?


Because this provides a *framework* to build loss functions for other
prediction types…


… and justifies least squares for real-valued regression models.


## Brief probability review

- Random variables, e.g. $x$ and $y$
- $\mathrm{Pr}(x)$ is a probability distribution over $x$
- $\mathrm{Pr}(x) \geq 0$ (a density can exceed 1; probabilities of discrete outcomes lie in $[0, 1]$)
- $\int_x \mathrm{Pr}(x)\, dx = 1$ or $\sum_i \mathrm{Pr}(x_i) = 1$
- $\mathrm{Pr}(x, y) = \mathrm{Pr}(x) \cdot \mathrm{Pr}(y)$ when $x$ and $y$ are independent
- $\mathrm{Pr}(x \mid y)\, \mathrm{Pr}(y) = \mathrm{Pr}(x, y) = \mathrm{Pr}(y \mid x)\, \mathrm{Pr}(x)$

See *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*, Appendix C
for more probability review.

A density is not a probability, so it can exceed 1: only its integral must
equal 1. A narrow normal distribution shows this, and its negative log density
at the peak is then negative. That is why a negative log-likelihood loss on
continuous data can be negative (as in the good-fit figure later, where it is
$-6.57$). Try a smaller $\sigma$.

In [ ]:
# Adapted from UDL notebook 5.1 (MIT)
import numpy as np

mu, sigma = 0.0, 0.3     # normal distribution parameters
y = 0.0                  # where to evaluate the density

def normal_pdf(y, mu, sigma):
    return np.exp(-(y - mu) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2)

density = normal_pdf(y, mu, sigma)
grid = np.linspace(mu - 10 * sigma, mu + 10 * sigma, 20001)
area = np.sum(normal_pdf(grid, mu, sigma)) * (grid[1] - grid[0])
print(f"Pr(y = {y} | mu = {mu}, sigma = {sigma}) = {density:.4f}   (> 1)")
print(f"-log Pr(y)                          = {-np.log(density):.4f}  (< 0)")
print(f"integral of the density             = {area:.4f}")

## Joint, marginal and conditional distributions

![](figs/03-Loss-Functions/conditional-distributions.png)


Marginal: integrate the joint over the other variable

$$
\mathrm{Pr}(x) = \int_y \mathrm{Pr}(x, y)\, dy
$$

Conditional: a slice of the joint, renormalized

$$
\mathrm{Pr}(x \mid y) = \frac{\mathrm{Pr}(x, y)}{\mathrm{Pr}(y)}
$$

$$
\int_x \mathrm{Pr}(x \mid y = 3.0)\, dx = 1
$$


## Continuous conditional distribution

For each input $x$ the model describes a full distribution over the output
$y$. For a continuous output (height) this is a density; the slides sweep $x$
through three ages.


Continuous output $y \in \mathbb{R}$

For every age $x$ the model gives a density $\mathrm{Pr}(y \mid x)$ over height.



![](figs/03-Loss-Functions/conditional-continuous-5.png)


## Discrete conditional distributions

For a discrete output the conditional distribution is a bar chart over the
classes. The slides sweep $x$ through 250, 980 and 1750 words for the spam
data, and through 0.35, 0.85 and 1.35 for the tail-length data.


Binary: $\mathrm{Pr}(y \mid x)$ is two bars


![](figs/03-Loss-Functions/conditional-binary-5.png)


Multiclass: $\mathrm{Pr}(y \mid x)$ is $K$ bars


![](figs/03-Loss-Functions/conditional-multiclass-4.png)


## Conditional distributions for different output types


![a) Real-valued output, b) class, c) count, d) direction. In each case the model gives a distribution $\mathrm{Pr}(y \mid x)$ shown at $x = 2$ and $x = 7$.](figs/03-Loss-Functions/conditional-distribution-types-4.png)


## Loss function and training

- Training dataset of $I$ pairs of input/output examples:
  $\{\mathbf{x}_i, \mathbf{y}_i\}_{i=1}^{I}$

- Loss function or cost function measures how bad the model is:

$$
L\Big[\boldsymbol{\phi}, \underbrace{\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]}_{\text{model}}, \underbrace{\{\mathbf{x}_i, \mathbf{y}_i\}_{i=1}^{I}}_{\text{train data}}\Big]
\qquad \text{or for short} \qquad L[\boldsymbol{\phi}]
$$

- Returns a scalar that is smaller when the model maps inputs to outputs better
- Training: find the parameters that minimize the loss:

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \Big[L[\boldsymbol{\phi}]\Big]
$$

(How to find the minimum is the next lecture: *gradient descent*.)


## Example: 1D linear regression loss function

![](figs/03-Loss-Functions/linear-regression-loss.png)


Loss function:

$$
\begin{aligned}
L[\boldsymbol{\phi}] &= \sum_{i=1}^{I} \left(\mathrm{f}[x_i, \boldsymbol{\phi}] - y_i\right)^2 \\
&= \sum_{i=1}^{I} \left(\phi_0 + \phi_1 x_i - y_i\right)^2
\end{aligned}
$$

"Least squares loss function"

Why *squared* error? And what do we do for a binary or a class label?


# Maximum likelihood

Instead of choosing a loss function directly, we let the model predict a
probability distribution over the output and choose the parameters that make
the observed training data most probable. This is maximum likelihood
estimation, which you met in DS701; here the twist is that a network predicts
the distribution's parameters.


## Maximum likelihood estimation

- In statistics, *maximum likelihood estimation (MLE)* is a method of
  *estimating the parameters* of an *assumed probability distribution*, *given
  some observed data*.
- This is achieved by *maximizing a likelihood function* so that, under the
  assumed statistical model, *the observed data is most probable*.

How do we do this with a network?

- ~~Model predicts output $y$ given input $\mathbf{x}$~~
- Model predicts a conditional probability distribution
  $\mathrm{Pr}(\mathbf{y} \mid \mathbf{x})$ over outputs $\mathbf{y}$ given inputs $\mathbf{x}$
- Define and minimize a loss function that makes the observed outputs have high probability


## How can a model predict a probability distribution? → Parametric models

1. Pick a known distribution (e.g., normal distribution) to model output $y$ with
   parameters $\boldsymbol{\theta}$

   e.g., the normal distribution $\boldsymbol{\theta} = \{\mu, \sigma^2\}$

   ![](figs/03-Loss-Functions/normal-distribution-peak.png)

2. Use the model to predict the parameters $\boldsymbol{\theta}$ of the probability distribution


## The likelihood of the training data

- We know we picked a good model and the right parameters when the joint
  conditional probability of the observed (training) outputs is high:

$$
\mathrm{Pr}(y_1, y_2, \ldots, y_I \mid x_1, x_2, \ldots, x_I)
$$

- Two simplifying assumptions:

$$
\mathrm{Pr}(y_1, y_2, \ldots, y_I \mid x_1, x_2, \ldots, x_I) = \underbrace{\prod_{i=1}^{I} \overbrace{\mathrm{Pr}(y_i \mid x_i)}^{\text{identically distributed}}}_{\text{independent}}
$$

- **Identically distributed**: the form of the probability distribution is the
  same for each input/output pair
- **Independent**: the joint factors into a product

*Independent and identically distributed (i.i.d.)*


## Maximum likelihood criterion

$$
\begin{aligned}
\hat{\boldsymbol{\phi}} &= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \mathbf{x}_i)\right] \\
&= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \boldsymbol{\theta}_i)\right] \\
&= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\right]
\end{aligned}
$$

- $\boldsymbol{\theta}_i$ are the parameters of the probability distribution
- $\boldsymbol{\phi}$ are the parameters of the neural network, e.g.
  $\boldsymbol{\theta}_i = \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}]$

When we consider this probability as a function of the parameters
$\boldsymbol{\phi}$, we call it a **likelihood**.

## Problem: a product of small numbers

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\right]
$$

- The terms in this product are often small (and the product underflows): each $\mathrm{Pr}(\cdot) \geq 0$, a density integrates to 1 and a discrete probability lies in $[0, 1]$
- The product might get so small that we *can't* easily represent it in fixed
  precision arithmetic

![](figs/03-Loss-Functions/log-monotonic.png)

The log function is monotonic: the maximum of the logarithm of a function is in
the same place as the maximum of the function.


The underflow is easy to trigger. A thousand samples from a standard normal,
each with a density around 0.2 to 0.4, already give a likelihood below the
smallest positive float64 (about $10^{-308}$); the sum of logs is unaffected.

In [ ]:
rng = np.random.default_rng(0)
I = 1000                                  # number of training examples
y = rng.normal(0.0, 1.0, size=I)
densities = normal_pdf(y, 0.0, 1.0)

print(f"product of {I} densities: {np.prod(densities)}")
print(f"sum of log densities:     {np.sum(np.log(densities)):.2f}")
print(f"=> likelihood = 10^{np.sum(np.log10(densities)):.1f}")

## Maximum log-likelihood → minimum negative log-likelihood

$$
\begin{aligned}
\hat{\boldsymbol{\phi}} &= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\right] \\
&= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\log\left[\prod_{i=1}^{I} \mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\right]\right] \\
&= \mathop{\mathrm{argmax}}_{\boldsymbol{\phi}} \left[\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right] \\
&= \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right] \\
&= \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \Big[L[\boldsymbol{\phi}]\Big]
\end{aligned}
$$

- Now it's a sum of terms, so it doesn't matter so much if the terms are small
- By convention, we minimize things (i.e., a loss), so we negate it: the
  **negative log-likelihood** loss

## Inference

- But now we predict a probability distribution
- We need an actual prediction (point estimate)
- Find the peak of the probability distribution (i.e., mean for normal)

$$
\hat{y} = \hat{\mu} = \mathop{\mathrm{argmax}}_{y}\Big[\mathrm{Pr}(y \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])\Big]
$$


![](figs/03-Loss-Functions/normal-distribution-peak.png)


# Recipe for loss functions

Maximum likelihood gives a general four-step recipe for building a loss
function for any kind of output. The rest of the lecture applies it to three
output types, and the recipe is repeated in the summary.


## Recipe for loss functions

1. Choose a suitable probability distribution $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta})$
   that is defined over the domain of the predictions $\mathbf{y}$ and has
   distribution parameters $\boldsymbol{\theta}$.
2. Set the machine learning model $\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ to
   predict one or more of these parameters so
   $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ and
   $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta}) = \mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}])$.
3. To train the model, find the network parameters $\hat{\boldsymbol{\phi}}$
   that minimize the negative log-likelihood loss function over the training
   dataset pairs $\{\mathbf{x}_i, \mathbf{y}_i\}$:
   $$
   \hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \big[L[\boldsymbol{\phi}]\big] = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right]. \qquad (5.6)
   $$
4. To perform inference for a new test example $\mathbf{x}$, return either the
   full distribution $\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])$
   or the maximum of this distribution.


# Example 1: univariate regression

The first example applies the recipe to a real-valued output and recovers the
least squares loss.


## Example 1: univariate regression

![](figs/03-Loss-Functions/regression-pipeline.png)

- Goal: predict a scalar output $y \in \mathbb{R}$ from the input $\mathbf{x}$

## Example 1: univariate regression (step 1)

> 1\. Choose a suitable probability distribution $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta})$
> that is defined over the domain of the predictions $\mathbf{y}$ and has
> distribution parameters $\boldsymbol{\theta}$.

- Predict scalar output: $y \in \mathbb{R}$
- Sensible probability distribution:
  - Normal distribution

$$
\mathrm{Pr}(y \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y - \mu)^2}{2\sigma^2}\right]
$$


![](figs/03-Loss-Functions/normal-distribution-examples.png)


## Example 1: univariate regression (step 2)

> 2\. Set the machine learning model $\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ to
> predict one or more of these parameters so
> $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ and
> $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta}) = \mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}])$.

$$
\mathrm{Pr}(y \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y - \mu)^2}{2\sigma^2}\right]
$$

In this case, just the mean:

$$
\mathrm{Pr}(y \mid \underbrace{\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]}_{\mu}, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y - \mathrm{f}[\mathbf{x}, \boldsymbol{\phi}])^2}{2\sigma^2}\right]
$$

Just learn the mean, $\mu$, and assume the variance is fixed.

## Example 1: univariate regression (step 3)

> 3\. To train the model, find the network parameters $\hat{\boldsymbol{\phi}}$
> that minimize the negative log-likelihood loss function over the training
> dataset pairs $\{\mathbf{x}_i, \mathbf{y}_i\}$:

$$
\begin{aligned}
L[\boldsymbol{\phi}] &= -\sum_{i=1}^{I} \log\left[\mathrm{Pr}(y_i \mid \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}], \sigma^2)\right] \\
&= -\sum_{i=1}^{I} \log\left[\frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y_i - \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}])^2}{2\sigma^2}\right]\right]
\end{aligned}
$$

You'll fill in the details in a homework:

- Log of products is sum of logs
- Log of exp() cancels each other out
- Ignore constants in minimization

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \big[L[\boldsymbol{\phi}]\big] = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[\sum_{i=1}^{I} (y_i - \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}])^2\right] \qquad \textbf{Least squares!}
$$


<!-- The SP26 slide names Homework 3; the number is left out because the homework schedule may change. -->

## Least squares vs. negative log-likelihood

With a fixed variance, a good fit gives a small least squares loss *and* a
small negative log-likelihood; a poor fit makes both large.

![a) Least squares; c) negative log-likelihood, for a line that fits the data well.](figs/03-Loss-Functions/least-squares-vs-nll-good-fit.png)

![b) Least squares; d) negative log-likelihood, for a line that fits the data poorly.](figs/03-Loss-Functions/least-squares-vs-nll-bad-fit.png)



The equivalence is exact, not approximate. With $\sigma$ fixed,
$L_{\text{NLL}} = \tfrac{I}{2}\log[2\pi\sigma^2] + L_{\text{LS}}/(2\sigma^2)$
is the least squares loss scaled and shifted, so both have their minimum at
the same $\boldsymbol{\phi}$ whatever $\sigma$ is. The cell evaluates both
losses for a line $\phi_0 + \phi_1 x$ on a grid of parameters and compares
their minimizers with the closed-form least squares solution.

In [ ]:
#| code-fold: true
#| fig-cap: "Left: toy data and the line that minimizes both losses. Right: the least squares loss and the Gaussian negative log-likelihood ($\\sigma = 0.2$) along $\\phi_1$, with $\\phi_0$ at its optimum. On their own axes the two curves are identical: one is an affine function of the other."
#| fig-alt: "Left: 20 orange data points rising from about 0.2 to 1.0 over x from 0 to 1, with a fitted blue line. Right: two U-shaped curves against slope phi_1, the least squares loss in blue on the left axis and the negative log-likelihood in orange on the right axis, drawn thick solid and dashed; the curves lie exactly on top of each other, with their minimum at a dashed vertical line near phi_1 = 0.71."
# Adapted from UDL notebook 5.1 (MIT)
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = rng.uniform(0, 1, size=20)
y = 0.2 + 0.8 * x + rng.normal(0, 0.15, size=20)

def least_squares(phi0, phi1):
    return np.sum((phi0 + phi1 * x - y) ** 2, axis=-1)

def gaussian_nll(phi0, phi1, sigma):
    r = phi0 + phi1 * x - y
    return np.sum(0.5 * np.log(2 * np.pi * sigma ** 2) + r ** 2 / (2 * sigma ** 2), axis=-1)

phi0_grid = np.linspace(-0.5, 1.0, 301)
phi1_grid = np.linspace(-0.5, 2.0, 501)
P0, P1 = np.meshgrid(phi0_grid, phi1_grid, indexing="ij")
P0, P1 = P0[..., None], P1[..., None]

def grid_argmin(loss):
    i, j = np.unravel_index(np.argmin(loss), loss.shape)
    return phi0_grid[i], phi1_grid[j]

A = np.column_stack([np.ones_like(x), x])
phi_ls = np.linalg.lstsq(A, y, rcond=None)[0]
print(f"closed-form least squares:   phi0 = {phi_ls[0]:.3f}, phi1 = {phi_ls[1]:.3f}")
print("grid argmin, least squares:  phi0 = {:.3f}, phi1 = {:.3f}".format(*grid_argmin(least_squares(P0, P1))))
for sigma in [0.05, 0.2, 1.0]:
    print(f"grid argmin, NLL sigma={sigma:<4}: " + "phi0 = {:.3f}, phi1 = {:.3f}".format(
        *grid_argmin(gaussian_nll(P0, P1, sigma))))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 3.2))
xs = np.linspace(0, 1, 2)
ax0.plot(x, y, "o", color="#eb6834", mec="#52514e", ms=5)
ax0.plot(xs, phi_ls[0] + phi_ls[1] * xs, color="#2a78d6")
ax0.set_xlabel("$x$"); ax0.set_ylabel("$y$")

phi1_line = np.linspace(0.2, 1.2, 200)
ax1.plot(phi1_line, least_squares(phi_ls[0], phi1_line[:, None]), color="#2a78d6", lw=4)
ax1.set_xlabel(r"$\phi_1$"); ax1.set_ylabel("least squares", color="#2a78d6")
ax1b = ax1.twinx()
ax1b.plot(phi1_line, gaussian_nll(phi_ls[0], phi1_line[:, None], 0.2), "--", color="#eb6834", lw=2)
ax1b.set_ylabel(r"NLL, $\sigma = 0.2$", color="#eb6834")
ax1.axvline(phi_ls[1], ls="--", color="#52514e")
fig.tight_layout()
plt.show()

## Example 1: univariate regression (step 4)

> 4\. To perform inference for a new test example $\mathbf{x}$, return either the
> full distribution $\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])$
> or the maximum of this distribution.

Full distribution:

$$
\mathrm{Pr}(y \mid \mathrm{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}], \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y - \mathrm{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])^2}{2\sigma^2}\right]
$$

Max probability:

$$
\hat{y} = \hat{\mu} = \mathrm{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}]
$$


![](figs/03-Loss-Functions/normal-distribution-peak.png)


## Estimating the variance

- Perhaps surprisingly, the variance term disappeared from the least squares
  loss: with $\sigma^2$ fixed, it is a constant that does not change the argmin.

- But we could learn it during training:

$$
\hat{\boldsymbol{\phi}}, \hat{\sigma}^2 = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}, \sigma^2} \left[-\sum_{i=1}^{I} \log\left[\frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y_i - \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}])^2}{2\sigma^2}\right]\right]\right]
$$

- Do gradient descent on both the model parameters, $\boldsymbol{\phi}$, and the
  variance, $\sigma^2$:

$$
\frac{\partial L}{\partial \boldsymbol{\phi}} \quad \text{and} \quad \frac{\partial L}{\partial \sigma^2}
$$


## Heteroscedastic regression

- We were assuming that the noise $\sigma^2$ is the same everywhere (homoscedastic).
- But we could make the noise a function of the data $\mathbf{x}$.
- Build a model with two outputs:

$$
\begin{aligned}
\mu &= \mathrm{f}_1[\mathbf{x}, \boldsymbol{\phi}] \\
\sigma^2 &= \mathrm{f}_2[\mathbf{x}, \boldsymbol{\phi}]^2
\end{aligned}
$$

($\mathrm{f}_2$ is squared so that $\sigma^2$ is positive.)

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \left(\log\left[\frac{1}{\sqrt{2\pi \mathrm{f}_2[\mathbf{x}_i, \boldsymbol{\phi}]^2}}\right] - \frac{(y_i - \mathrm{f}_1[\mathbf{x}_i, \boldsymbol{\phi}])^2}{2\mathrm{f}_2[\mathbf{x}_i, \boldsymbol{\phi}]^2}\right)\right]
$$

## Heteroscedastic regression: a two-output network

![a) Network with one output $\mu$ and fixed $\sigma$; b) its prediction with a constant $\pm 2\sigma$ band. c) Network with two outputs $\mu$ and $\sigma$; d) its prediction with a band that varies with $x$.](figs/03-Loss-Functions/heteroscedastic-regression.png)

Takeaways: *least squares* is the right loss under a normal model with fixed
variance; the best prediction is the predicted mean; we can also estimate a
global or local variance.

Both variance models on toy data whose noise grows with $x$,
$\sigma(x) = 0.05 + 0.3x$. To isolate the variance, the mean
$\mu(x) = 0.5\sin(2\pi x)$ is taken as known. Setting
$\partial L / \partial \sigma^2 = 0$ for the homoscedastic model gives
$\hat{\sigma}^2 = \frac{1}{I}\sum_i (y_i - \mu_i)^2$, the mean squared residual;
the sweep on the left confirms it. A noise model that tracks $x$ explains the
data better and has a lower negative log-likelihood.

In [ ]:
#| code-fold: true
#| fig-cap: "Left: homoscedastic negative log-likelihood as a function of $\\sigma$, minimized at the root mean squared residual. Right: the data with the $\\pm 2\\sigma$ band of the best constant $\\sigma$ (gray) and of the heteroscedastic noise model (blue)."
#| fig-alt: "Left: a curve of negative log-likelihood against sigma from 0.05 to 0.6 that falls steeply and then rises slowly, with its minimum marked by a dashed line near sigma = 0.22. Right: 200 data points scattered around a sine wave, tight at small x and spread out at large x; a gray band of constant width and a blue band that widens with x."
# Adapted from UDL notebook 5.1 (MIT)
rng = np.random.default_rng(0)
I = 200
x = np.sort(rng.uniform(0, 1, size=I))
mu = 0.5 * np.sin(2 * np.pi * x)          # f_1: the predicted mean (taken as known)
sigma_x = 0.05 + 0.3 * x                  # f_2: the true noise level at each x
y = mu + sigma_x * rng.normal(size=I)

def nll(y, mu, sigma):
    return np.sum(0.5 * np.log(2 * np.pi * sigma ** 2) + (y - mu) ** 2 / (2 * sigma ** 2))

sigma_hat = np.sqrt(np.mean((y - mu) ** 2))   # closed-form MLE
sigmas = np.linspace(0.05, 0.6, 500)
sweep = np.array([nll(y, mu, s) for s in sigmas])
print(f"closed-form MLE sigma = {sigma_hat:.4f}, sweep minimum at sigma = {sigmas[sweep.argmin()]:.4f}")
print(f"NLL, homoscedastic (sigma = {sigma_hat:.3f}): {nll(y, mu, sigma_hat):8.2f}")
print(f"NLL, heteroscedastic sigma(x):          {nll(y, mu, sigma_x):8.2f}")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 3.2))
ax0.plot(sigmas, sweep, color="#2a78d6")
ax0.axvline(sigma_hat, ls="--", color="#52514e")
ax0.set_xlabel(r"$\sigma$"); ax0.set_ylabel("negative log-likelihood")
ax1.fill_between(x, mu - 2 * sigma_hat, mu + 2 * sigma_hat, color="0.85", label=r"constant $\hat\sigma$")
ax1.fill_between(x, mu - 2 * sigma_x, mu + 2 * sigma_x, color="#2a78d6", alpha=0.3, label=r"$\sigma(x)$")
ax1.plot(x, y, "o", color="#eb6834", ms=2.5)
ax1.plot(x, mu, color="#52514e")
ax1.set_xlabel("$x$"); ax1.set_ylabel("$y$"); ax1.legend(loc="lower left", fontsize=8)
fig.tight_layout()
plt.show()

# Example 2: binary classification

For a binary label the output is no longer a real number, so the recipe starts
from a different distribution and gives a different loss.


## Example 2: binary classification

![](figs/03-Loss-Functions/binary-classification-pipeline.png)

- Goal: predict which of two classes $y \in \{0, 1\}$ the input $\mathbf{x}$ belongs to

## Example 2: binary classification (step 1)

> 1\. Choose a suitable probability distribution $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta})$
> that is defined over the domain of the predictions $\mathbf{y}$ and has
> distribution parameters $\boldsymbol{\theta}$.

- Domain: $y \in \{0, 1\}$
- Bernoulli distribution
- One parameter $\lambda \in [0,1]$

$$
\mathrm{Pr}(y \mid \lambda) = \begin{cases} 1 - \lambda & \quad y = 0 \\ \lambda & \quad y = 1 \end{cases}
$$

or

$$
\mathrm{Pr}(y \mid \lambda) = (1 - \lambda)^{1 - y} \cdot \lambda^{y}
$$


![](figs/03-Loss-Functions/bernoulli-distribution.png)


## Example 2: binary classification (step 2)

> 2\. Set the machine learning model $\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ to
> predict one or more of these parameters so
> $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ and
> $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta}) = \mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}])$.

Problem:

- Output of neural network can be anything
- Parameter $\lambda \in [0,1]$

Solution:

- Pass through the logistic sigmoid function that maps "anything" to $[0,1]$:

$$
\mathrm{sig}[z] = \frac{1}{1 + \exp[-z]}
$$


![](figs/03-Loss-Functions/logistic-sigmoid.png)


Substituting $\lambda = \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]]$ into the Bernoulli:

$$
\mathrm{Pr}(y \mid \mathbf{x}) = (1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]])^{1 - y} \cdot \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]]^{y}
$$


## Example 2: binary classification (step 3)

> 3\. To train the model, find the network parameters $\hat{\boldsymbol{\phi}}$
> that minimize the negative log-likelihood loss function over the training
> dataset pairs $\{\mathbf{x}_i, \mathbf{y}_i\}$:

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \big[L[\boldsymbol{\phi}]\big] = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right]. \qquad (5.6)
$$

$$
\mathrm{Pr}(y \mid \mathbf{x}) = (1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]])^{1 - y} \cdot \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]]^{y}
$$

$$
L[\boldsymbol{\phi}] = \sum_{i=1}^{I} \Big( -(1 - y_i) \log\big[1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\big] - y_i \log\big[\mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\big] \Big)
$$

Also called the **binary cross-entropy loss**, because it is what the
cross-entropy calculation gives for a Bernoulli model -- discussed later.

## Example 2: binary classification (step 4)

> 4\. To perform inference for a new test example $\mathbf{x}$, return either the
> full distribution $\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])$
> or the maximum of this distribution.

![a) Network output $\mathrm{f}[x, \boldsymbol{\phi}]$; b) passed through the sigmoid; c) the resulting Bernoulli parameter $\lambda$ and $1 - \lambda$ as functions of $x$. The inset shows the Bernoulli distribution at one input.](figs/03-Loss-Functions/binary-classification-sigmoid-model.png)

- Choose $y = 1$ where $\lambda$ is greater than 0.5, otherwise 0
- And we get a probability estimate, or "confidence value"!

Takeaways: *binary cross-entropy* is the loss; threshold $\lambda$ to get a
prediction; the model also gives a probability.

# Example 3: multiclass classification

With $K$ classes, the Bernoulli distribution is replaced by the categorical
distribution and the sigmoid by the softmax.


## Example 3: multiclass classification

![](figs/03-Loss-Functions/multiclass-classification-pipeline.png)

- Goal: predict which of $K$ classes $y \in \{1, 2, \ldots, K\}$ the input $\mathbf{x}$ belongs to

## Example 3: multiclass classification (step 1)

> 1\. Choose a suitable probability distribution $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta})$
> that is defined over the domain of the predictions $\mathbf{y}$ and has
> distribution parameters $\boldsymbol{\theta}$.

- Domain: $y \in \{1, 2, \ldots, K\}$
- **Categorical distribution**
- $K$ parameters $\lambda_k \in [0,1]$
- $\sum_k \lambda_k = 1$

$$
\mathrm{Pr}(y = k) = \lambda_k
$$


![](figs/03-Loss-Functions/categorical-distribution.png)


## Example 3: multiclass classification (step 2)

> 2\. Set the machine learning model $\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ to
> predict one or more of these parameters so
> $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ and
> $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta}) = \mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}])$.

Problem:

- Output of neural network can be anything
- Parameters $\lambda_k \in [0,1]$, sum to one

Solution:

- Pass the $K$ outputs through a function that maps "anything" to $[0,1]$ and sums to one: the **softmax**


$$
\mathrm{softmax}_k[\mathbf{z}] = \frac{\exp[z_k]}{\sum_{k'=1}^{K} \exp[z_{k'}]}
$$


$$
\mathrm{Pr}(y = k \mid \mathbf{x}) = \mathrm{softmax}_k\big[\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]\big]
$$

## Example 3: multiclass classification (step 3)

> 3\. To train the model, find the network parameters $\hat{\boldsymbol{\phi}}$
> that minimize the negative log-likelihood loss function over the training
> dataset pairs $\{\mathbf{x}_i, \mathbf{y}_i\}$:

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \big[L[\boldsymbol{\phi}]\big] = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right]. \qquad (5.6)
$$

$$
\mathrm{softmax}_k[\mathbf{z}] = \frac{\exp[z_k]}{\sum_{k'=1}^{K} \exp[z_{k'}]}
$$

$$
\begin{aligned}
L[\boldsymbol{\phi}] &= -\sum_{i=1}^{I} \log\Big[\mathrm{softmax}_{y_i}\big[\mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}]\big]\Big] \\
&= -\sum_{i=1}^{I} \left( \mathrm{f}_{y_i}[\mathbf{x}_i, \boldsymbol{\phi}] - \log\left[\sum_{k=1}^{K} \exp\big[\mathrm{f}_k[\mathbf{x}_i, \boldsymbol{\phi}]\big]\right] \right)
\end{aligned}
$$

*Multiclass cross-entropy loss*

The second line above is also how the loss should be *computed*. Evaluating
the softmax first and then taking its log overflows (`exp(1000)` is `inf`) or
takes `log(0)` once the logits are large. The **log-sum-exp** trick subtracts
the largest logit first,
$\log\sum_k e^{z_k} = m + \log\sum_k e^{z_k - m}$ with $m = \max_k z_k$, so no
exponent is ever positive. Binary cross-entropy has the same fix: in terms of
the logit $z$, $-\log[\mathrm{sig}[z]] = \log[1 + e^{-z}]$, computed as
$\max(-z, 0) + \log[1 + e^{-|z|}]$. This is why PyTorch's
`BCEWithLogitsLoss` and `CrossEntropyLoss` take logits, not probabilities.

In [ ]:
# Adapted from UDL notebooks 5.2 and 5.3 (MIT)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_naive(z, y):
    lam = sigmoid(z)
    return -(1 - y) * np.log(1 - lam) - y * np.log(lam)

def bce_stable(z, y):
    # -log sig(z) = softplus(-z), -log(1 - sig(z)) = softplus(z)
    softplus = lambda t: np.maximum(t, 0) + np.log1p(np.exp(-np.abs(t)))
    return (1 - y) * softplus(z) + y * softplus(-z)

def ce_naive(z, k):
    p = np.exp(z) / np.sum(np.exp(z))
    return -np.log(p[k])

def ce_stable(z, k):
    m = np.max(z)
    return -(z[k] - m - np.log(np.sum(np.exp(z - m))))

with np.errstate(all="ignore"):
    print("binary cross-entropy, label y = 0")
    for z in [2.0, 20.0, 40.0, 800.0]:
        print(f"  logit {z:6.0f}: naive = {bce_naive(z, 0):10.4f}   stable = {bce_stable(z, 0):10.4f}")
    print("multiclass cross-entropy, true class k = 1")
    for z in [np.array([2.0, 1.0, -1.0]), np.array([1000.0, 0.0, -1000.0])]:
        print(f"  logits {z}: naive = {ce_naive(z, 1):10.4f}   stable = {ce_stable(z, 1):10.4f}")

## Example 3: multiclass classification (step 4)

> 4\. To perform inference for a new test example $\mathbf{x}$, return either the
> full distribution $\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])$
> or the maximum of this distribution.

![a) Three network outputs $\mathrm{f}_1$, $\mathrm{f}_2$, $\mathrm{f}_3$; b) after the softmax, the class probabilities $\lambda_1$, $\lambda_2$, $\lambda_3$, which sum to one at every $x$.](figs/03-Loss-Functions/multiclass-softmax-model.png)

- Choose the class with the largest probability
- We also get a probability or "confidence" for every class

# Other types of data

The same recipe works for any output type once we pick a distribution defined
over the right domain.


## Other data types

| Data Type | Domain | Distribution | Use |
|---|---|---|---|
| univariate, continuous, unbounded | $y \in \mathbb{R}$ | univariate normal | regression |
| univariate, continuous, unbounded | $y \in \mathbb{R}$ | Laplace or t-distribution | robust regression |
| univariate, continuous, unbounded | $y \in \mathbb{R}$ | mixture of Gaussians | multimodal regression |
| univariate, continuous, bounded below | $y \in \mathbb{R}^{+}$ | exponential or gamma | predicting magnitude |
| univariate, continuous, bounded | $y \in [0, 1]$ | beta | predicting proportions |
| multivariate, continuous, unbounded | $\mathbf{y} \in \mathbb{R}^{K}$ | multivariate normal | multivariate regression |
| univariate, continuous, circular | $y \in (-\pi, \pi]$ | von Mises | predicting direction |
| univariate, discrete, binary | $y \in \{0, 1\}$ | Bernoulli | binary classification |
| univariate, discrete, bounded | $y \in \{1, 2, \ldots, K\}$ | categorical | multiclass classification |
| univariate, discrete, bounded below | $y \in [0, 1, 2, 3, \ldots]$ | Poisson | predicting event counts |
| multivariate, discrete, permutation | $\mathbf{y} \in \mathrm{Perm}[1, 2, \ldots, K]$ | Plackett-Luce | ranking |

: Distributions for loss functions for different prediction types (Prince, *Understanding Deep Learning*, Figure 5.11).

## Other distributions

![**Gaussian**: $y \in \mathbb{R}$ Regression](figs/03-Loss-Functions/distribution-gaussian.png)

![**Laplace**: $y \in \mathbb{R}$ Robust Regression](figs/03-Loss-Functions/distribution-laplace.png)

![**Mixture of Gaussians**: $y \in \mathbb{R}$ Multimodal Regression](figs/03-Loss-Functions/distribution-mixture-of-gaussians.png)

![**Gamma**: $y \in \mathbb{R}^{+}$ Predict Magnitude](figs/03-Loss-Functions/distribution-gamma.png)


![**Beta**: $y \in [0,1]$ Predict Proportions](figs/03-Loss-Functions/distribution-beta.png)

![**Von Mises**: $y \in (-\pi, \pi]$ Predict Directions](figs/03-Loss-Functions/distribution-von-mises.png){fig-alt="Plot of a von Mises distribution over angles from -pi to 2 pi: a peak at 0 that repeats at 2 pi."}

![**Poisson**: $y \in [0, 1, 2, \ldots]$ Predict Event Counts](figs/03-Loss-Functions/distribution-poisson.png)


# Multiple outputs

When the model has several outputs, we treat them as independent so the
negative log-likelihood becomes a sum over outputs as well as over examples.


## Multiple outputs

- Treat each output $y_d$ as *independent*:

$$
\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}]) = \prod_{d} \mathrm{Pr}(y_{id} \mid \mathbf{f}_d[\mathbf{x}_i, \boldsymbol{\phi}])
$$

where $\mathbf{f}_d[\mathbf{x}, \boldsymbol{\phi}]$ is the $d^{th}$ set of network outputs

- Negative log-likelihood becomes a sum of terms:

$$
L[\boldsymbol{\phi}] = -\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big] = -\sum_{i=1}^{I} \sum_{d} \log\Big[\mathrm{Pr}(y_{id} \mid \mathbf{f}_d[\mathbf{x}_i, \boldsymbol{\phi}])\Big]
$$

where $y_{id}$ is the $d^{th}$ output of the $i^{th}$ training example

## Example 4: multivariate regression

![](figs/03-Loss-Functions/multivariate-regression-pipeline.png)

- Goal: predict a multivariate target $\mathbf{y} \in \mathbb{R}^{D_o}$

## Example 4: multivariate regression (independent outputs)

- Solution: treat each dimension independently, with a normal distribution
  over each one

$$
\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\mu}, \sigma^2) = \prod_{d=1}^{D_o} \mathrm{Pr}(y_d \mid \mu_d, \sigma^2) = \prod_{d=1}^{D_o} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y_d - \mu_d)^2}{2\sigma^2}\right]
$$

- Make a network with $D_o$ outputs to predict the means

$$
\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}], \sigma^2) = \prod_{d=1}^{D_o} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left[-\frac{(y_d - \mathrm{f}_d[\mathbf{x}, \boldsymbol{\phi}])^2}{2\sigma^2}\right]
$$

- What if the outputs vary in magnitude (e.g., weight in kilos and height in
  meters)? One dimension dominates the loss.
  - Could learn a separate variance for each output…
  - …or rescale the targets before training, and rescale the outputs in the
    opposite way at inference


# Cross entropy

Cross-entropy loss comes from information theory. This section introduces
entropy and the KL divergence, then shows that minimizing the KL divergence
between the data distribution and the model gives the same negative
log-likelihood loss as before. Cross-entropy is not a different loss, just a
different route to the same one.


## Information theory and entropy

- **Claude Shannon:** the "father of information theory," was an American
  mathematician, electrical engineer, and cryptographer
- **Theory of Communication:** In his landmark 1948 paper, "A Mathematical
  Theory of Communication," Shannon introduced a formal framework for the
  transmission, processing, and storage of information.
- **Information Theory:** Quantified information, allowing for the measurement
  of information content in messages, which is crucial for data compression,
  error detection and correction, and more.
- **Concept of Information Entropy:** introduced entropy as a measure of the
  uncertainty or randomness in a set of possible messages, providing a limit on
  the best possible lossless compression of any communication.

$$
H(x) = -\sum_x \mathrm{Pr}(x) \log_2(\mathrm{Pr}(x))
$$


![Claude Shannon, and the title page of *The Mathematical Theory of Communication* (Shannon and Weaver, 1949).](figs/03-Loss-Functions/claude-shannon.png)


## Entropy is a measure of *surprise* or *uncertainty on average*

Binary event $x \in \{0, 1\}$, true with probability $p$:

$$
\begin{aligned}
H(x) &= -\sum_x \mathrm{Pr}(x)\, \log_2\big(\mathrm{Pr}(x)\big) \\
&= -p \log_2(p) - (1 - p) \log_2(1 - p)
\end{aligned}
$$

![](figs/03-Loss-Functions/binary-entropy.png)

Peaks at 50/50.


Randomly pick a ball from the box. Which box has the higher entropy?

![](figs/03-Loss-Functions/entropy-ball-boxes.png)


<!-- The SP26 slide links an in-class Piazza poll; omitted. -->

## Connection to deep learning

**Cross-entropy loss**

- If a neural network predicts **(0.25, 0.25, 0.25, 0.25)** for four possible
  classes, **high entropy** → uncertain.
- If it predicts **(0.99, 0.01, 0, 0)**, **low entropy** → confident.

**Regularization and overfitting**

- A **high-entropy** model makes diverse predictions → good for exploration.
- A **low-entropy** model could be overconfident → might overfit.

"Raise the temperature in LLMs."

## Cross entropy: comparing the data distribution with the model

- So far we defined loss functions that minimize the negative log-likelihood.
- *Cross-entropy loss* is the usual name in neural network training.
- We will show that it is the same thing: it measures the difference between
  the empirical distribution of the data, $q(y)$, and the model distribution
  $\mathrm{Pr}(y \mid \boldsymbol{\theta})$.

![a) The empirical data distribution $q(y)$; b) a model distribution $\mathrm{Pr}(y \mid \boldsymbol{\theta})$. Training minimizes the KL divergence between them.](figs/03-Loss-Functions/empirical-vs-model-distribution.png)

**Kullback-Leibler divergence** -- a measure of distance between probability distributions

## Empirical distribution: a collection of samples

![](figs/03-Loss-Functions/empirical-distribution-deltas.png)


Each sample is represented by a shifted Dirac delta function.

$$
\int \delta[x - x_0]\, dx = 1
$$

$$
\int f[x]\, \delta[x - x_0]\, dx = f[x_0]
$$

So we say the empirical distribution is

$$
q(y) = \frac{1}{I} \sum_{i=1}^{I} \delta[\, y - y_i]
$$

which will be a helpful formulation in a moment.


## Kullback-Leibler (KL) divergence

How much a model distribution, $p(z)$, differs from the true (data)
distribution, $q(z)$:

$$
\begin{aligned}
D_{KL}\big[q(z) \,\|\, p(z)\big] &= \int_{-\infty}^{\infty} q(z) \log \frac{q(z)}{p(z)}\, dz \\
&= \int_{-\infty}^{\infty} q(z)\, \log[q(z)]\, dz - \int_{-\infty}^{\infty} q(z)\, \log[p(z)]\, dz
\end{aligned}
$$

![](figs/03-Loss-Functions/kl-gaussians.png)

<!-- TODO(verify): KL notebook to be re-hosted in the SP27 materials repo; link it from the materials_base variable. -->


![](figs/03-Loss-Functions/kl-integrand.png)

In the plots the true distribution is $P(x) = \mathcal{N}(0, 1)$ and the model is
$Q(x)$ with mean 1 and standard deviation 2, so $P$ plays the role of $q$ and
$Q$ of $p$; $D_{KL}[P \,\|\, Q]$ is the area under the integrand: 0.4431.


Two normal distributions have a closed-form KL divergence,
$$
D_{KL}\big[\mathcal{N}(\mu_1, \sigma_1^2) \,\|\, \mathcal{N}(\mu_2, \sigma_2^2)\big]
= \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}.
$$
The cell checks it against numerical integration of the integrand plotted
above, then swaps the arguments: the KL divergence is not symmetric, so it is
not a true distance.

In [ ]:
mu_p, sigma_p = 0.0, 1.0      # P: the true distribution
mu_q, sigma_q = 1.0, 2.0      # Q: the model

def kl_numeric(m1, s1, m2, s2):
    z = np.linspace(-30, 30, 600001)
    p, q = normal_pdf(z, m1, s1), normal_pdf(z, m2, s2)
    return np.sum(p * np.log(p / q)) * (z[1] - z[0])

def kl_closed(m1, s1, m2, s2):
    return np.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5

print(f"D_KL[P || Q]: numerical = {kl_numeric(mu_p, sigma_p, mu_q, sigma_q):.4f}, "
      f"closed form = {kl_closed(mu_p, sigma_p, mu_q, sigma_q):.4f}")
print(f"D_KL[Q || P]: numerical = {kl_numeric(mu_q, sigma_q, mu_p, sigma_p):.4f}, "
      f"closed form = {kl_closed(mu_q, sigma_q, mu_p, sigma_p):.4f}")

## Derivation

Training dataset as a collection of Dirac delta functions:

$$
q(y) = \frac{1}{I} \sum_{i=1}^{I} \delta[y - y_i],
$$

Minimize the KL divergence from the data distribution to the model:

$$
\begin{aligned}
\hat{\boldsymbol{\theta}} &= \mathop{\mathrm{argmin}}_{\boldsymbol{\theta}} \left[\int_{-\infty}^{\infty} q(y) \log\big[q(y)\big]\, dy - \int_{-\infty}^{\infty} q(y) \log\big[\mathrm{Pr}(y \mid \boldsymbol{\theta})\big]\, dy\right]
\end{aligned}
$$


The 1st term does not depend on $\boldsymbol{\theta}$ (it is the negative
entropy of the data). What remains is the **cross-entropy** between $q$ and the
model:

$$
= \mathop{\mathrm{argmin}}_{\boldsymbol{\theta}} \left[-\int_{-\infty}^{\infty} q(y) \log\big[\mathrm{Pr}(y \mid \boldsymbol{\theta})\big]\, dy\right],
$$


## Derivation (cont.)

Substituting for $q(y)$:

$$
\hat{\boldsymbol{\theta}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\theta}} \left[-\int_{-\infty}^{\infty} \left(\frac{1}{I} \sum_{i=1}^{I} \delta[y - y_i]\right) \log\big[\mathrm{Pr}(y \mid \boldsymbol{\theta})\big]\, dy\right]
$$


Property of the Dirac delta function:

$$
= \mathop{\mathrm{argmin}}_{\boldsymbol{\theta}} \left[-\frac{1}{I} \sum_{i=1}^{I} \log\big[\mathrm{Pr}(y_i \mid \boldsymbol{\theta})\big]\right]
$$


$1/I$ is just a constant, so ignore:

$$
= \mathop{\mathrm{argmin}}_{\boldsymbol{\theta}} \left[-\sum_{i=1}^{I} \log\big[\mathrm{Pr}(y_i \mid \boldsymbol{\theta})\big]\right].
$$


The model is predicting $\boldsymbol{\theta}$ → **Negative log-likelihood!**

$$
\hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\big[\mathrm{Pr}(y_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\big]\right]
$$


# Summary

Every loss in this lecture is a negative log-likelihood under a probability
model of the output; the choice of distribution is the only thing that changes.


## Summary

- Reconsidered loss functions as fitting a parametric probability model
  $\mathrm{Pr}(\mathbf{y} \mid \mathbf{x})$ to the training data
- Maximum likelihood: choose the parameters that make the training data most
  probable; the i.i.d. assumption turns the likelihood into a product, and the
  log turns it into a sum
- Four-step recipe: (1) pick a distribution over the output domain, (2) have
  the model predict its parameters, (3) train by minimizing the negative
  log-likelihood, (4) infer by returning the distribution or its maximum
- Univariate regression: normal with fixed variance → *least squares*; the
  variance can also be learned, globally or as a function of $\mathbf{x}$
  (heteroscedastic)
- Binary classification: Bernoulli with sigmoid → *binary cross-entropy*;
  threshold $\lambda$ at 0.5 to predict
- Multiclass classification: categorical with softmax → *cross-entropy*; the
  argmax is the prediction
- Other output types: pick the distribution for the domain (UDL Figure 5.11);
  multiple outputs: treat as independent and sum the losses
- Cross entropy from information theory: minimizing the KL divergence from the
  empirical distribution to the model is the same as minimizing the negative
  log-likelihood
- Next: find the parameters that minimize the loss → *training the model*

## Summary: recipe for loss functions

1. Choose a suitable probability distribution $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta})$
   that is defined over the domain of the predictions $\mathbf{y}$ and has
   distribution parameters $\boldsymbol{\theta}$.
2. Set the machine learning model $\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ to
   predict one or more of these parameters so
   $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$ and
   $\mathrm{Pr}(\mathbf{y} \mid \boldsymbol{\theta}) = \mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}])$.
3. To train the model, find the network parameters $\hat{\boldsymbol{\phi}}$
   that minimize the negative log-likelihood loss function over the training
   dataset pairs $\{\mathbf{x}_i, \mathbf{y}_i\}$:
   $$
   \hat{\boldsymbol{\phi}} = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \big[L[\boldsymbol{\phi}]\big] = \mathop{\mathrm{argmin}}_{\boldsymbol{\phi}} \left[-\sum_{i=1}^{I} \log\Big[\mathrm{Pr}(\mathbf{y}_i \mid \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])\Big]\right]
   $$
4. To perform inference for a new test example $\mathbf{x}$, return either the
   full distribution $\mathrm{Pr}(\mathbf{y} \mid \mathbf{f}[\mathbf{x}, \hat{\boldsymbol{\phi}}])$
   or the maximum of this distribution.